In [1]:
# Setup
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

# Work with cleaned data
df = pd.read_csv("../data/diabetic_data_cleaned.csv")
print(df.shape)

(100114, 87)


In [2]:
feature_cols = (
    ["age_encoded", "max_glu_serum_encoded", "A1Cresult_encoded",
     "number_outpatient", "number_emergency", "number_inpatient",
     "time_in_hospital", "num_lab_procedures", "num_medications", "number_diagnoses"]
    + ["diag_1_group_Diabetes", "diag_1_group_Digestive", "diag_1_group_Genitourinary",
       "diag_1_group_Injury", "diag_1_group_Musculoskeletal", "diag_1_group_Neoplasms",
       "diag_1_group_Other", "diag_1_group_Respiratory"]
    + ["discharge_disposition_id_2", "discharge_disposition_id_3", "discharge_disposition_id_4",
       "discharge_disposition_id_5", "discharge_disposition_id_6", "discharge_disposition_id_7",
       "discharge_disposition_id_8", "discharge_disposition_id_9", "discharge_disposition_id_10",
       "discharge_disposition_id_12", "discharge_disposition_id_13", "discharge_disposition_id_14",
       "discharge_disposition_id_15", "discharge_disposition_id_16", "discharge_disposition_id_17",
       "discharge_disposition_id_18", "discharge_disposition_id_22", "discharge_disposition_id_23",
       "discharge_disposition_id_24", "discharge_disposition_id_25", "discharge_disposition_id_27",
       "discharge_disposition_id_28"]
    + ["race_Asian", "race_Caucasian", "race_Hispanic", "race_Other", "race_Unknown"]
    + ["gender_Male", "gender_Unknown/Invalid"]
)

X = df[feature_cols]
y = df["readmitted_30d"]

# Check X and y
print(X.shape)
print(y.value_counts())

(100114, 47)
readmitted_30d
0    88757
1    11357
Name: count, dtype: int64


In [3]:
# Split into train\test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(80091, 47) (20023, 47)
readmitted_30d
0    0.886554
1    0.113446
Name: proportion, dtype: float64
readmitted_30d
0    0.88658
1    0.11342
Name: proportion, dtype: float64


In [4]:
# Scale features before fitting
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)     # To avoid data leakage

In [5]:
# Fit model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled, y_train)

print("Model trained")

Model trained


In [6]:
# Evaluate model
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

print("AUC:", roc_auc_score(y_test, y_pred_proba))       # Classifying ability
print("Precision:", precision_score(y_test, y_pred))     # Percentage of predicted patients who are actually readmitted
print("Recall:", recall_score(y_test, y_pred))           # Percentage of actually readmitted patients predicted by model
print("Confusion Matrix:")                               # Matrix: [[TN, FP], [FN, TP]]
print(confusion_matrix(y_test, y_pred))

AUC: 0.6585919877746114
Precision: 0.180119466993414
Recall: 0.5178335535006605
Confusion Matrix:
[[12399  5353]
 [ 1095  1176]]


In [7]:
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]     # model.coef_ is a 2D array
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values("abs_coefficient", ascending=False)     # To sort by magnitude, not sign

# See which features matter most
print(coefficients.head(15))

                         feature  coefficient  abs_coefficient
5               number_inpatient     0.382647         0.382647
34   discharge_disposition_id_22     0.192556         0.192556
19    discharge_disposition_id_3     0.148435         0.148435
21    discharge_disposition_id_5     0.111831         0.111831
9               number_diagnoses     0.094150         0.094150
18    discharge_disposition_id_2     0.086093         0.086093
28   discharge_disposition_id_13    -0.078134         0.078134
17      diag_1_group_Respiratory    -0.076523         0.076523
4               number_emergency     0.069088         0.069088
33   discharge_disposition_id_18     0.066126         0.066126
39   discharge_disposition_id_28     0.065051         0.065051
22    discharge_disposition_id_6     0.064362         0.064362
31   discharge_disposition_id_16    -0.059884         0.059884
32   discharge_disposition_id_17    -0.058742         0.058742
14  diag_1_group_Musculoskeletal    -0.057252         0

In [8]:
mapping = pd.read_csv("../data/IDS_mapping.csv")
print(mapping)

   admission_type_id                                        description
0                  1                                          Emergency
1                  2                                             Urgent
2                  3                                           Elective
3                  4                                            Newborn
4                  5                                      Not Available
..               ...                                                ...
62                22   Transfer from hospital inpt/same fac reslt in...
63                23                          Born inside this hospital
64                24                         Born outside this hospital
65                25            Transfer from Ambulatory Surgery Center
66                26                              Transfer from Hospice

[67 rows x 2 columns]
